<a href="https://colab.research.google.com/github/Ladecodes1/formative-3-group/blob/main/part2_bayes_imdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 2: Bayesian Probability on IMDb Movie Reviews
Goal: for each keyword, compute P(Positive | keyword) using Bayes' theorem.
Dataset: 50,000 IMDb reviews (25,000 positive, 25,000 negative).
Keywords: excellent, wonderful, superb, brilliant (positive); worst, awful, waste, boring (negative).

## Step 1: Load the data
We read the CSV with Python's built-in csv module. Each review is turned into a set of lowercase words, so a word is counted once per review.

In [ ]:
import csv
import re

DATA_PATH = "IMDB Dataset.csv"


def load_reviews(path):
    """Read the CSV and return a list of (set_of_words, label) pairs."""
    reviews = []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            text = row["review"].lower().replace("<br />", " ")
            words = set(re.findall(r"[a-z']+", text))
            reviews.append((words, row["sentiment"].strip().lower()))
    return reviews


reviews = load_reviews(DATA_PATH)
print("Total reviews:", len(reviews))
print("Example label:", reviews[0][1])
print("Some words from review 1:", list(reviews[0][0])[:8])

Total reviews: 50000
Example label: positive
Some words from review 1: ['get', 'main', 'scuffles', 'developed', 'is', 'guards', 'inwards', 'hardcore']


## Step 2: Counting helper and the prior
One reusable function counts reviews by keyword and/or label. The prior P(Positive) is positive reviews ÷ total reviews.

In [ ]:
def count_reviews(reviews, keyword=None, label=None):
    """Count reviews, optionally only those containing a keyword and/or with a given label."""
    return sum(
        1
        for words, lab in reviews
        if (keyword is None or keyword in words)
        and (label is None or lab == label)
    )


total = count_reviews(reviews)
n_positive = count_reviews(reviews, label="positive")
prior = n_positive / total

print("Total reviews:", total)
print("Positive reviews:", n_positive)
print("Prior P(Positive):", prior)

Total reviews: 50000
Positive reviews: 25000
Prior P(Positive): 0.5


## Step 2: Counting helper and the prior
One reusable function counts reviews by keyword and/or label. The prior P(Positive) is positive reviews ÷ total reviews.

In [ ]:
keyword = "excellent"

# Counts (each one reuses count_reviews with different filters)
n_keyword_and_positive = count_reviews(reviews, keyword=keyword, label="positive")
n_keyword = count_reviews(reviews, keyword=keyword)

# The four probabilities
prior = n_positive / total
likelihood = n_keyword_and_positive / n_positive
marginal = n_keyword / total
posterior = likelihood * prior / marginal

print("Keyword:", keyword)
print("Reviews containing it:", n_keyword)
print("Of those, positive:", n_keyword_and_positive)
print()
print("Prior      P(Positive)             =", prior)
print("Likelihood P(excellent | Positive) =", likelihood)
print("Marginal   P(excellent)            =", marginal)
print("Posterior  P(Positive | excellent) =", posterior)

# Sanity check: the direct count should give the same answer
print("Direct check:", n_keyword_and_positive / n_keyword)

Keyword: excellent
Reviews containing it: 3549
Of those, positive: 2865

Prior      P(Positive)             = 0.5
Likelihood P(excellent | Positive) = 0.1146
Marginal   P(excellent)            = 0.07098
Posterior  P(Positive | excellent) = 0.8072696534234995
Direct check: 0.8072696534234995


## Step 4: Bayes' theorem as a function, applied to all eight keywords
P(Positive | keyword) = P(keyword | Positive) × P(Positive) ÷ P(keyword)

In [ ]:
def bayes_posterior(prior, likelihood, marginal):
    """Bayes' theorem: P(A|B) = P(B|A) * P(A) / P(B)"""
    return likelihood * prior / marginal


def analyse_keyword(reviews, keyword):
    """Return the four probabilities for P(Positive | keyword)."""
    total = count_reviews(reviews)
    n_positive = count_reviews(reviews, label="positive")
    n_keyword = count_reviews(reviews, keyword=keyword)
    n_both = count_reviews(reviews, keyword=keyword, label="positive")

    prior = n_positive / total
    likelihood = n_both / n_positive
    marginal = n_keyword / total
    posterior = bayes_posterior(prior, likelihood, marginal)

    # Sanity check: Bayes must match the direct count
    assert abs(posterior - n_both / n_keyword) < 1e-9

    return {
        "keyword": keyword,
        "prior": prior,
        "likelihood": likelihood,
        "marginal": marginal,
        "posterior": posterior,
    }


POSITIVE_KEYWORDS = ["excellent", "wonderful", "superb", "brilliant"]
NEGATIVE_KEYWORDS = ["worst", "awful", "waste", "boring"]

results = [analyse_keyword(reviews, k) for k in POSITIVE_KEYWORDS + NEGATIVE_KEYWORDS]

print(f"{'Keyword':<12}{'P(Pos)':>9}{'P(kw|Pos)':>12}{'P(kw)':>9}{'P(Pos|kw)':>12}")
for r in results:
    print(f"{r['keyword']:<12}{r['prior']:>9.4f}{r['likelihood']:>12.4f}"
          f"{r['marginal']:>9.4f}{r['posterior']:>12.4f}")

Keyword        P(Pos)   P(kw|Pos)    P(kw)   P(Pos|kw)
excellent      0.5000      0.1146   0.0710      0.8073
wonderful      0.5000      0.0902   0.0555      0.8120
superb         0.5000      0.0410   0.0240      0.8557
brilliant      0.5000      0.0634   0.0417      0.7606
worst          0.5000      0.0164   0.0886      0.0924
awful          0.5000      0.0112   0.0575      0.0977
waste          0.5000      0.0070   0.0507      0.0687
boring         0.5000      0.0236   0.0610      0.1938


## Interpretation
Positive keywords all have posteriors well above 0.5 (superb is strongest at 0.856, brilliant weakest at 0.761).
Negative keywords all have posteriors well below 0.5 (waste is lowest at 0.069, boring highest at 0.194), so reviews containing them are unlikely to be positive.